# GitHub Open-Source Repository Recommendation System

## 1. Executive Summary

**Problem.** Popularity-led discovery does not reliably surface open-source projects aligned with a developer's public technical interests. **Data.** This reduced, real-public-data run contains 5 developers, 796 repositories, and 449 owned/starred interactions collected from the GitHub REST API; bounded README/file checks use GitHub's public raw-content host. **Method.** Weighted training-only developer profiles feed TF–IDF cosine similarity, language/topic matching, repository activity/quality, and calibrated popularity; random, popularity, language, content, and hybrid rankers are compared. **Evaluation.** All five users use chronological holdouts, with five most-recent test repositories held out per user.

**Measured result.** `language` ranked first by test NDCG@10 at **0.1357**, with Hit Rate@10 **0.40**, Recall@10 **0.08**, and sampled-catalog coverage **7.01%**. The content model's NDCG@10 was 0.0476, compared with random 0.0000, popularity 0.0678, and language-only 0.1357; the notebook reports each comparison rather than assuming content or hybrid wins. **Main limitation.** Five selected users and a bounded catalog support an engineering demonstration, not population-level claims.


## 2. Business Problem

Developers face a discovery problem: repository popularity is easy to see, but relevance to their skills, technical interests, desired activity level, and contribution readiness is not. The business question is whether public GitHub activity and repository metadata can recommend more relevant projects than random, global-popularity, or language-only rules.

- **Target user:** a developer exploring or contributing to public open source.
- **Unit of analysis:** a developer–repository candidate pair.
- **Decision supported:** which unseen public repositories to place in the top recommendation positions.
- **Analytical question:** how accurately can held-out public interactions be retrieved in the top K?
- **Success criteria:** rank-aware gains over baselines, useful catalog coverage/diversity, reproducibility, and explanations grounded in calculated features.
- **Constraints:** public data only, API limits, missing star timestamps in the documented HTML fallback, and no claim of ranking all GitHub.
- **Ethics/privacy:** only public professional activity is used; no private activity, sensitive-trait inference, or automated contribution decisions.


## 3. Project Objectives

The objective is an auditable content-plus-quality recommender that accepts a sampled GitHub username, excludes repositories already observed in that developer's profile, ranks an eligible catalog, explains each score, and compares against random, popularity, and language baselines using leak-resistant holdouts.


## 4. Data Sources

The primary source is the official GitHub REST API (API version `2022-11-28`): repository search, public user profiles, and timestamped public starred repositories. README and contribution-file content uses GitHub's public raw-content host. The first unauthenticated pass reached the hourly core limit; cached retry after reset completed all five histories through the API, so the committed modelling sample uses no HTML-derived interactions. The collection manifest preserves queries, failures, row counts, and provenance.


## 5. Data Limitations

This is a convenience sample, not a representative sample of GitHub. The run contains five deliberately varied public developers, up to the 100 most recent API-visible timestamped stars per user, and a language-stratified candidate catalog. Missing byte-level language distributions and issue-label checks are retained as missing. All five committed histories use chronological evaluation. Stars indicate interest, not necessarily contribution intent.


## 6. Environment Setup

The workflow uses pinned Python packages, deterministic seeds, `pathlib` paths, the non-interactive Matplotlib backend, and only relative project paths. A GitHub token is optional and is read only from `GITHUB_TOKEN`.


In [1]:
from __future__ import annotations

import base64
import hashlib
import html
import json
import logging
import os
import random
import re
import time
import warnings
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scipy
import sklearn
from dotenv import load_dotenv
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.preprocessing import MinMaxScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)
print({"python_runtime": "3.11+", "pandas": pd.__version__, "numpy": np.__version__, "scikit_learn": sklearn.__version__, "matplotlib": matplotlib.__version__})


Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\pandy\\.matplotlib\\fontlist-v390.json.matplotlib-lock'


{'python_runtime': '3.11+', 'pandas': '2.2.3', 'numpy': '2.1.3', 'scikit_learn': '1.5.2', 'matplotlib': '3.9.3'}


C:\Users\pandy\Documents\Codex\2026-08-01\files-mentioned-by-the-user-you-2\work\.venv-github-rec\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 7. Configuration and Reproducibility

`RANDOM_SEED` controls every randomized holdout and baseline. The analysis date is anchored to collection time so activity scores reproduce later. `FULL_COLLECTION=False` loads the committed real sample; authenticated users can call the client in sections 8–9 and increase collection scope without changing modelling code.


In [2]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
load_dotenv()

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "sample").exists():
    candidate = PROJECT_ROOT / "End to end data science project" / "GitHub Open-Source Repository Recommendation System"
    if candidate.exists():
        PROJECT_ROOT = candidate
for directory in ["data/raw", "data/processed", "data/sample", "outputs/figures", "models"]:
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)

SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = PROJECT_ROOT / "models"
manifest = json.loads((SAMPLE_DIR / "collection_manifest.json").read_text(encoding="utf-8"))
ANALYSIS_AS_OF = pd.Timestamp(manifest["collected_at_utc"]).floor("D")
FULL_COLLECTION = False
print({"project_root": str(PROJECT_ROOT), "seed": RANDOM_SEED, "analysis_as_of": str(ANALYSIS_AS_OF), "mode": manifest["mode"]})


{'project_root': 'C:\\Users\\pandy\\Documents\\Codex\\2026-08-01\\files-mentioned-by-the-user-you-2\\work\\Data-Science-\\End to end data science project\\GitHub Open-Source Repository Recommendation System', 'seed': 42, 'analysis_as_of': '2026-08-02 00:00:00+00:00', 'mode': 'reduced unauthenticated sample'}


## 8. GitHub API Client

The reusable client below centralizes authentication, headers, pagination, timeouts, status checks, exponential retries, primary/secondary rate-limit handling, raw-response caching, incremental collection, duplicate prevention, logging, and intermediate saves. A permanent failure is logged and returned as missing rather than silently converted into data.


In [3]:
class GitHubAPIClient:
    # Cached, retrying client for public GitHub REST collection.

    API_ROOT = "https://api.github.com"

    def __init__(self, cache_dir: Path, token: str | None = None, timeout: int = 30, retries: int = 3):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.timeout, self.retries = timeout, retries
        self.session = requests.Session()
        self.logger = logging.getLogger("github_api")
        self.failures: list[dict[str, Any]] = []
        self.headers = self.github_headers(token)

    @staticmethod
    def github_headers(token: str | None = None) -> dict[str, str]:
        headers = {
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "github-open-source-recommender-portfolio",
        }
        if token:
            headers["Authorization"] = f"Bearer {token}"
        return headers

    def _cache_file(self, url: str, params: Mapping[str, Any] | None, accept: str | None) -> Path:
        key = json.dumps([url, dict(params or {}), accept or ""], sort_keys=True)
        return self.cache_dir / f"{hashlib.sha256(key.encode()).hexdigest()}.json"

    def request_json(self, path_or_url: str, params: Mapping[str, Any] | None = None, accept: str | None = None) -> Any:
        url = path_or_url if path_or_url.startswith("http") else f"{self.API_ROOT}{path_or_url}"
        cache_file = self._cache_file(url, params, accept)
        if cache_file.exists():
            return json.loads(cache_file.read_text(encoding="utf-8"))
        headers = dict(self.headers)
        if accept:
            headers["Accept"] = accept
        for attempt in range(self.retries + 1):
            try:
                response = self.session.get(url, params=params, headers=headers, timeout=self.timeout)
                if response.status_code == 403:
                    remaining, reset = response.headers.get("X-RateLimit-Remaining"), response.headers.get("X-RateLimit-Reset")
                    if remaining == "0" and reset:
                        wait_seconds = max(1, int(reset) - int(time.time()) + 2)
                        if wait_seconds <= 120 and attempt < self.retries:
                            self.logger.warning("Primary limit reached; waiting %s seconds", wait_seconds)
                            time.sleep(wait_seconds)
                            continue
                    if attempt < self.retries:  # secondary limit: bounded backoff
                        time.sleep(min(5 * (2 ** attempt), 30))
                        continue
                if response.status_code in {429, 500, 502, 503, 504} and attempt < self.retries:
                    time.sleep(min(2 * (2 ** attempt), 20))
                    continue
                response.raise_for_status()
                payload = response.json()
                cache_file.write_text(json.dumps(payload), encoding="utf-8")
                return payload
            except (requests.RequestException, ValueError) as exc:
                if attempt == self.retries:
                    failure = {"url": url, "params": dict(params or {}), "error": str(exc)}
                    self.failures.append(failure)
                    self.logger.error("Permanent failure: %s", failure)
                    return None
                time.sleep(min(2 * (2 ** attempt), 20))
        return None

    def paginate(self, path: str, params: Mapping[str, Any] | None = None, max_pages: int | None = None, accept: str | None = None) -> list[dict[str, Any]]:
        results: list[dict[str, Any]] = []
        base_params = dict(params or {})
        base_params.setdefault("per_page", 100)
        page = 1
        while max_pages is None or page <= max_pages:
            payload = self.request_json(path, {**base_params, "page": page}, accept=accept)
            if payload is None:
                break
            batch = payload.get("items", []) if isinstance(payload, dict) else payload
            if not batch:
                break
            results.extend(batch)
            if len(batch) < int(base_params["per_page"]):
                break
            page += 1
        deduped = {item.get("id", hashlib.sha1(json.dumps(item, sort_keys=True).encode()).hexdigest()): item for item in results}
        return list(deduped.values())

    def collect_user(self, username: str) -> dict[str, Any] | None:
        return self.request_json(f"/users/{username}")

    def collect_repositories(self, query: str, max_pages: int = 1) -> list[dict[str, Any]]:
        return self.paginate("/search/repositories", {"q": query, "sort": "updated"}, max_pages=max_pages)

    def collect_starred(self, username: str, max_pages: int = 1) -> list[dict[str, Any]]:
        return self.paginate(f"/users/{username}/starred", max_pages=max_pages, accept="application/vnd.github.star+json")

    def collect_repository_languages(self, full_name: str) -> dict[str, int] | None:
        return self.request_json(f"/repos/{full_name}/languages")

    def collect_readme(self, full_name: str) -> str | None:
        payload = self.request_json(f"/repos/{full_name}/readme")
        if not payload or not payload.get("content"):
            return None
        return base64.b64decode(payload["content"]).decode("utf-8", errors="replace")

    def collect_topics(self, full_name: str) -> list[str]:
        payload = self.request_json(f"/repos/{full_name}") or {}
        return payload.get("topics", [])

    def check_repository_files(self, full_name: str, paths: Sequence[str]) -> dict[str, bool | None]:
        checks: dict[str, bool | None] = {}
        for path in paths:
            payload = self.request_json(f"/repos/{full_name}/contents/{path}")
            checks[path] = True if payload else None
        return checks

    def collect_issue_labels(self, full_name: str) -> list[str]:
        return [str(item.get("name", "")).lower() for item in self.paginate(f"/repos/{full_name}/labels", max_pages=1)]

    @staticmethod
    def save_intermediate(records: Iterable[Mapping[str, Any]], path: Path) -> None:
        rows = list(records)
        path.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).drop_duplicates().to_csv(path, index=False)

api_client = GitHubAPIClient(PROJECT_ROOT / "data" / "raw" / "cache", os.getenv("GITHUB_TOKEN"))
print("Reusable client ready; live calls disabled in committed sample mode.")


Reusable client ready; live calls disabled in committed sample mode.


## 9. Public Data Collection

The committed sample was collected once with the client design above and is loaded locally to avoid consuming reviewers' rate limits. The manifest is the audit trail. Authenticated scaling should expand developers and repository pages incrementally; raw caches remain ignored by Git.


In [4]:
developers_raw = pd.read_csv(SAMPLE_DIR / "developers.csv")
repositories_raw = pd.read_csv(SAMPLE_DIR / "repositories.csv", low_memory=False)
languages_raw = pd.read_csv(SAMPLE_DIR / "repository_languages.csv")
interactions_raw = pd.read_csv(SAMPLE_DIR / "interactions.csv")
print("Manifest rows:", manifest["rows"])
print("Sources:", repositories_raw.get("metadata_source", pd.Series(["github_rest_api"])).value_counts(dropna=False).to_dict())
print("Recorded API failures:", len(manifest.get("api_failures", [])))


Manifest rows: {'developers': 5, 'repositories': 796, 'repository_languages': 772, 'interactions': 449}
Sources: {'github_rest_api': 1}
Recorded API failures: 0


## 10. Raw Data Validation

Schema assertions fail loudly if required columns disappear. IDs, timestamps, and booleans are normalized only after the raw counts are retained for the quality report.


In [5]:
REQUIRED = {
    "developers": {"developer_id", "username", "account_created_at", "followers", "following", "public_repo_count"},
    "repositories": {"repository_id", "full_name", "repository_url", "owner", "name", "description", "readme_text", "primary_language", "topics", "stars", "forks", "watchers", "open_issues", "created_at", "updated_at", "pushed_at", "license", "archived", "disabled", "fork", "repository_size", "has_readme", "has_license", "has_contributing", "has_code_of_conduct", "has_good_first_issue", "has_help_wanted"},
    "repository_languages": {"repository_id", "language", "language_bytes", "language_percentage"},
    "interactions": {"developer_id", "repository_id", "interaction_type", "interaction_weight", "interaction_timestamp", "timestamp_available"},
}
raw_tables = {"developers": developers_raw, "repositories": repositories_raw, "repository_languages": languages_raw, "interactions": interactions_raw}
for name, required in REQUIRED.items():
    missing = required - set(raw_tables[name].columns)
    assert not missing, f"{name} missing columns: {sorted(missing)}"
print({name: {"rows": len(df), "columns": len(df.columns)} for name, df in raw_tables.items()})


{'developers': {'rows': 5, 'columns': 6}, 'repositories': {'rows': 796, 'columns': 30}, 'repository_languages': {'rows': 772, 'columns': 5}, 'interactions': {'rows': 449, 'columns': 6}}


## 11. Data Cleaning

Cleaning standardizes types, strips impossible IDs, removes exact duplicate entities/interactions, parses timestamps with UTC semantics, and preserves missing metadata. No archived/disabled/forked records are silently deleted here; candidate eligibility is a separate, auditable decision.


In [6]:
def parse_bool(series: pd.Series) -> pd.Series:
    mapping = {"true": True, "false": False, "1": True, "0": False}
    return series.map(lambda x: mapping.get(str(x).strip().lower(), x if isinstance(x, bool) else pd.NA)).astype("boolean")

developers = developers_raw.copy()
repositories = repositories_raw.copy()
repository_languages = languages_raw.copy()
interactions = interactions_raw.copy()

for frame, id_col in [(developers, "developer_id"), (repositories, "repository_id"), (repository_languages, "repository_id"), (interactions, "developer_id"), (interactions, "repository_id")]:
    frame[id_col] = pd.to_numeric(frame[id_col], errors="coerce")
for column in ["followers", "following", "public_repo_count"]:
    developers[column] = pd.to_numeric(developers[column], errors="coerce")
for column in ["stars", "forks", "watchers", "open_issues", "repository_size"]:
    repositories[column] = pd.to_numeric(repositories[column], errors="coerce")
for column in ["archived", "disabled", "fork", "has_readme", "has_license", "has_contributing", "has_code_of_conduct", "has_good_first_issue", "has_help_wanted"]:
    repositories[column] = parse_bool(repositories[column])
interactions["timestamp_available"] = parse_bool(interactions["timestamp_available"])
interactions["interaction_weight"] = pd.to_numeric(interactions["interaction_weight"], errors="coerce")

for column in ["account_created_at"]:
    developers[column] = pd.to_datetime(developers[column], errors="coerce", utc=True)
for column in ["created_at", "updated_at", "pushed_at"]:
    repositories[column] = pd.to_datetime(repositories[column], errors="coerce", utc=True)
interactions["interaction_timestamp"] = pd.to_datetime(interactions["interaction_timestamp"], errors="coerce", utc=True)

developers = developers.dropna(subset=["developer_id", "username"]).drop_duplicates("developer_id").copy()
repositories = repositories.dropna(subset=["repository_id", "full_name", "repository_url"]).drop_duplicates("repository_id").copy()
repository_languages = repository_languages.dropna(subset=["repository_id", "language"]).drop_duplicates(["repository_id", "language"]).copy()
interactions = interactions.dropna(subset=["developer_id", "repository_id", "interaction_type"]).drop_duplicates(["developer_id", "repository_id", "interaction_type"]).copy()
for frame, columns in [(developers, ["developer_id"]), (repositories, ["repository_id"]), (repository_languages, ["repository_id"]), (interactions, ["developer_id", "repository_id"])]:
    for column in columns:
        frame[column] = frame[column].astype("int64")

known_repo_ids = set(repositories["repository_id"])
known_developer_ids = set(developers["developer_id"])
interactions = interactions[interactions["repository_id"].isin(known_repo_ids) & interactions["developer_id"].isin(known_developer_ids)].copy()
print({"developers": len(developers), "repositories": len(repositories), "languages": len(repository_languages), "interactions": len(interactions)})


{'developers': 5, 'repositories': 796, 'languages': 772, 'interactions': 449}


## 12. Data Quality Summary

Each check reports scope, impact, action, and reason. “Retained/flagged” means the observation remains available so missingness and API limits remain visible. Candidate exclusions happen only when safety or usefulness justifies them.


In [7]:
quality_rows: list[dict[str, Any]] = []
def add_quality(check: str, evaluated: int, affected: int, action: str, reason: str) -> None:
    quality_rows.append({"check_name": check, "records_evaluated": int(evaluated), "records_affected": int(affected), "percentage_affected": round(100 * affected / evaluated, 2) if evaluated else 0.0, "action_taken": action, "reason": reason})

add_quality("missing developer profile values", developers.size, int(developers.isna().sum().sum()), "retained and flagged", "HTML fallback does not expose every REST profile field")
add_quality("duplicate developers", len(developers_raw), int(developers_raw.duplicated("developer_id").sum()), "deduplicated by developer_id", "one public account per ID")
add_quality("duplicate repositories", len(repositories_raw), int(repositories_raw.duplicated("repository_id").sum()), "deduplicated by repository_id", "search and stars overlap")
add_quality("duplicate interactions", len(interactions_raw), int(interactions_raw.duplicated(["developer_id", "repository_id", "interaction_type"]).sum()), "deduplicated by typed pair", "prevent double weighting")
add_quality("invalid repository timestamps", len(repositories), int(repositories[["created_at", "updated_at", "pushed_at"]].isna().all(axis=1).sum()), "retained and flagged", "HTML fallback timestamps are unavailable")
add_quality("empty descriptions", len(repositories), int(repositories["description"].fillna("").str.strip().eq("").sum()), "retained; README/topics may supply content", "description is optional on GitHub")
add_quality("empty README text", len(repositories), int(repositories["readme_text"].fillna("").str.strip().eq("").sum()), "retained and quality-scored", "raw checks were rate/bandwidth bounded")
add_quality("missing topics", len(repositories), int(repositories["topics"].fillna("[]").isin(["[]", "", "nan"]).sum()), "retained", "topics are optional")
add_quality("missing primary language", len(repositories), int(repositories["primary_language"].isna().sum()), "retained", "empty or metadata-light repositories can lack language")
add_quality("missing licenses", len(repositories), int(repositories["license"].isna().sum()), "retained and quality-scored", "license presence is a ranking signal")
add_quality("archived repositories", len(repositories), int(repositories["archived"].fillna(False).sum()), "excluded from candidates", "not open for active contribution")
add_quality("disabled repositories", len(repositories), int(repositories["disabled"].fillna(False).sum()), "excluded from candidates", "unusable candidate")
add_quality("forked repositories", len(repositories), int(repositories["fork"].fillna(False).sum()), "excluded from candidates", "recommend canonical projects")
days_since_push_dq = (ANALYSIS_AS_OF - repositories["pushed_at"]).dt.days
add_quality("extremely inactive repositories (>5 years)", len(repositories), int((days_since_push_dq > 1825).sum()), "excluded from candidates", "stale projects reduce contribution value")
add_quality("extremely popular repositories (top 1%)", len(repositories), int((repositories["stars"] >= repositories["stars"].quantile(.99)).sum()), "retained; tested in robustness", "measure popularity bias rather than hide it")
activity_counts = interactions.groupby("developer_id").size()
add_quality("users with insufficient history (<20)", len(developers), int((developers["developer_id"].map(activity_counts).fillna(0) < 20).sum()), "excluded from offline evaluation only", "profiles/holdouts would be unstable")
metadata_empty = repositories["description"].fillna("").str.strip().eq("") & repositories["readme_text"].fillna("").str.strip().eq("") & repositories["primary_language"].isna()
add_quality("repositories with insufficient metadata", len(repositories), int(metadata_empty.sum()), "excluded from candidates", "content scoring is not meaningful")
add_quality("API failures", manifest.get("requests_made", 0), len(manifest.get("api_failures", [])), "logged in collection manifest", "no failed response becomes a row")
rate_failures = sum("rate limit" in str(item.get("error", "")).lower() for item in manifest.get("api_failures", []))
add_quality("rate-limit interruptions", manifest.get("requests_made", 0), rate_failures, "HTML fallback or missing values", "unauthenticated GitHub core limit")
add_quality("uneven user activity", len(developers), int((activity_counts.max() - activity_counts.min()) > 20), "reported; macro-average metrics", "avoid prolific users dominating evaluation")
add_quality("missing language byte distributions", len(repository_languages), int(repository_languages["language_bytes"].isna().sum()), "primary-language fallback retained", "per-repository language calls exceed reduced-mode budget")

data_quality = pd.DataFrame(quality_rows)
data_quality.to_csv(OUTPUT_DIR / "data_quality_summary.csv", index=False)
developers.to_csv(PROCESSED_DIR / "developers.csv", index=False)
repositories.to_csv(PROCESSED_DIR / "repositories.csv", index=False)
repository_languages.to_csv(PROCESSED_DIR / "repository_languages.csv", index=False)
interactions.to_csv(PROCESSED_DIR / "interactions.csv", index=False)
print(data_quality.to_string(index=False))


                                check_name  records_evaluated  records_affected  percentage_affected                               action_taken                                                   reason
          missing developer profile values                 30                 0                 0.00                       retained and flagged   HTML fallback does not expose every REST profile field
                      duplicate developers                  5                 0                 0.00               deduplicated by developer_id                                one public account per ID
                    duplicate repositories                796                 0                 0.00              deduplicated by repository_id                                 search and stars overlap
                    duplicate interactions                449                 0                 0.00                 deduplicated by typed pair                                 prevent double weigh

## 13. Exploratory Data Analysis

All charts use Matplotlib, one figure per file. Technically, log axes prevent a handful of famous repositories/users from flattening the distributions. Product interpretation: the catalog is long-tailed and metadata coverage is uneven, so popularity must be calibrated and missing readiness signals treated as uncertainty—not failure.


In [8]:
def save_bar(values: pd.Series, title: str, xlabel: str, ylabel: str, filename: str, rotation: int = 0) -> None:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    values.plot(kind="bar", ax=ax, color="#3569b7")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotation)
    fig.tight_layout(); fig.savefig(FIGURE_DIR / filename, dpi=150); plt.close(fig)

def save_hist(values: pd.Series, title: str, xlabel: str, filename: str, log_x: bool = False) -> None:
    clean = pd.to_numeric(values, errors="coerce").dropna()
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.hist(np.log1p(clean) if log_x else clean, bins=25, color="#2a9d8f", edgecolor="white")
    ax.set_title(title); ax.set_xlabel(f"log(1 + {xlabel})" if log_x else xlabel); ax.set_ylabel("Count")
    fig.tight_layout(); fig.savefig(FIGURE_DIR / filename, dpi=150); plt.close(fig)

save_hist(developers["public_repo_count"], "Developer public-repository counts", "public repositories", "developer_public_repos.png", True)
save_hist(developers["followers"], "Developer follower distribution", "followers", "developer_followers.png", True)
save_bar(interactions.groupby(interactions["developer_id"].map(developers.set_index("developer_id")["username"])).size().sort_values(ascending=False), "Interactions per developer", "developer", "interactions", "developer_interactions.png", 30)
save_hist(repositories["stars"], "Repository star distribution", "stars", "repository_stars.png", True)
save_hist(repositories["forks"], "Repository fork distribution", "forks", "repository_forks.png", True)
save_hist(repositories["open_issues"], "Open-issue distribution", "open issues", "repository_open_issues.png", True)

top_languages = repositories["primary_language"].fillna("Missing").value_counts().head(12)
save_bar(top_languages, "Most common primary languages", "language", "repositories", "repository_languages.png", 35)
interaction_types = interactions["interaction_type"].value_counts()
save_bar(interaction_types, "Observed public interaction types", "interaction type", "interactions", "interaction_types.png")
metadata_rates = pd.Series({
    "README checked+present": repositories["has_readme"].fillna(False).mean(),
    "License": repositories["has_license"].fillna(False).mean(),
    "Contributing guide": repositories["has_contributing"].fillna(False).mean(),
    "Code of conduct": repositories["has_code_of_conduct"].fillna(False).mean(),
    "Good-first-issue known": repositories["has_good_first_issue"].fillna(False).mean(),
    "Help-wanted known": repositories["has_help_wanted"].fillna(False).mean(),
}) * 100
save_bar(metadata_rates, "Repository metadata/readiness availability", "signal", "repositories (%)", "metadata_availability.png", 40)

repo_interaction_counts = interactions.groupby("repository_id").size()
save_hist(repo_interaction_counts, "Interactions per observed repository", "interactions", "interactions_per_repository.png")
sorted_stars = repositories["stars"].fillna(0).sort_values(ascending=False).to_numpy()
cumulative = np.cumsum(sorted_stars) / max(sorted_stars.sum(), 1)
fig, ax = plt.subplots(figsize=(8, 4.8)); ax.plot(np.arange(1, len(cumulative)+1) / len(cumulative) * 100, cumulative * 100, color="#e76f51")
ax.axhline(80, color="grey", linestyle="--"); ax.set_title("Popularity concentration (stars)"); ax.set_xlabel("Catalog ranked by popularity (%)"); ax.set_ylabel("Cumulative stars (%)")
fig.tight_layout(); fig.savefig(FIGURE_DIR / "popularity_concentration.png", dpi=150); plt.close(fig)

interaction_repo_share = interactions["repository_id"].nunique() / len(repositories)
matrix_density = len(interactions) / max(len(developers) * len(repositories), 1)
top_20_count = max(1, int(.2 * len(sorted_stars)))
top_20_star_share = sorted_stars[:top_20_count].sum() / max(sorted_stars.sum(), 1)
print({"figures_saved": len(list(FIGURE_DIR.glob("*.png"))), "user_item_density": round(matrix_density, 4), "catalog_with_interactions": round(interaction_repo_share, 4), "top_20pct_star_share": round(top_20_star_share, 4), "interpretation": "strong popularity concentration supports a capped popularity weight"})


{'figures_saved': 12, 'user_item_density': 0.1128, 'catalog_with_interactions': 0.5603, 'top_20pct_star_share': np.float64(0.9578), 'interpretation': 'strong popularity concentration supports a capped popularity weight'}


## 14. Developer Profile Construction

Profiles use only training interactions during validation/test scoring. Weighted repository vectors use the modelling assumptions `owned=4`, `forked=3`, `starred=2`, and `contributed=5`; only observed interaction types enter this sample. All committed REST stars are timestamped and split chronologically. The code retains a seeded leave-five-out fallback for future missing-timestamp samples and never labels that fallback chronological.


In [9]:
INTERACTION_WEIGHT_ASSUMPTIONS = {"contributed": 5.0, "owned": 4.0, "forked": 3.0, "starred": 2.0}
interactions["split"] = "train"
split_method: dict[int, str] = {}
for developer_id, group in interactions[interactions["interaction_type"] == "starred"].groupby("developer_id"):
    group = group.drop_duplicates("repository_id")
    if len(group) < 20:
        continue
    timestamped = group["timestamp_available"].fillna(False) & group["interaction_timestamp"].notna()
    if timestamped.all():
        order = group.sort_values("interaction_timestamp").index.to_numpy()
        split_method[int(developer_id)] = "chronological"
    else:
        order = group.index.to_numpy().copy()
        np.random.default_rng(RANDOM_SEED + int(developer_id)).shuffle(order)
        split_method[int(developer_id)] = "reproducible_leave_five_out"
    interactions.loc[order[-5:], "split"] = "test"
    interactions.loc[order[-10:-5], "split"] = "validation"

eval_users = sorted([int(dev) for dev, g in interactions.groupby("developer_id") if (g["split"] == "train").sum() >= 10 and (g["split"] == "test").sum() >= 1])
split_summary = interactions.groupby([interactions["developer_id"].map(developers.set_index("developer_id")["username"]), "split"]).size().unstack(fill_value=0)
print(split_summary)
print("Split methods:", {developers.set_index("developer_id").loc[k, "username"]: v for k, v in split_method.items()})


split         test  train  validation
developer_id                         
gaearon          5     90           5
hadley           5     90           5
jakevdp          5     39           5
karpathy         5     90           5
sindresorhus     5     90           5
Split methods: {'hadley': 'chronological', 'sindresorhus': 'chronological', 'karpathy': 'chronological', 'jakevdp': 'chronological', 'gaearon': 'chronological'}


## 15. Repository Text Processing

Repository text combines description, bounded README text, topics, primary language, and observed major languages. Cleaning lowercases, removes URLs/HTML/badges and obvious installation boilerplate, collapses whitespace, preserves technical tokens such as `c++`, `c#`, and framework names, and caps README input at 8,000 characters. No aggressive stemming is used.


In [10]:
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.I)
HTML_RE = re.compile(r"<[^>]+>")
BADGE_RE = re.compile(r"!\[[^\]]*\]\([^)]*(?:badge|shield)[^)]*\)", re.I)
BOILERPLATE_RE = re.compile(r"(?:table of contents|installation|getting started|license)\s*[:\n]+", re.I)

def clean_repository_text(value: Any, max_chars: int = 8_000) -> str:
    text = "" if pd.isna(value) else str(value)[:max_chars]
    text = BADGE_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = HTML_RE.sub(" ", html.unescape(text))
    text = BOILERPLATE_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text.lower()).strip()

def parse_topics(value: Any) -> list[str]:
    if pd.isna(value):
        return []
    try:
        parsed = json.loads(str(value))
        return [str(item).lower() for item in parsed] if isinstance(parsed, list) else []
    except json.JSONDecodeError:
        return [item.strip().lower() for item in str(value).split(",") if item.strip()]

repositories["topic_list"] = repositories["topics"].map(parse_topics)
repositories["description_clean"] = repositories["description"].map(clean_repository_text)
repositories["readme_clean"] = repositories["readme_text"].map(clean_repository_text)
repositories["text_description"] = (repositories["description_clean"] + " " + repositories["topic_list"].map(" ".join) + " " + repositories["primary_language"].fillna("").str.lower()).str.strip()
repositories["text_full"] = (repositories["text_description"] + " " + repositories["readme_clean"]).str.strip()
print(repositories[["full_name", "text_full"]].assign(text_chars=lambda x: x["text_full"].str.len()).sort_values("text_chars", ascending=False).head(3)[["full_name", "text_chars"]])


                            full_name  text_chars
450          hjanuschka/pi-multi-pass        7606
608     bluesky-social/feed-generator        7594
536  huggingface/pytorch-image-models        7546


## 16. Repository Feature Engineering

Content uses TF–IDF. Independent ranking components include content cosine similarity, primary-language preference, topic preference, recent activity, contribution-oriented quality, calibrated popularity, and beginner friendliness. Numeric transforms are fitted on the eligible item catalog only; they use no held-out interaction labels.


In [11]:
repositories["days_since_push"] = (ANALYSIS_AS_OF - repositories["pushed_at"]).dt.days.clip(lower=0)
repositories["repository_age_days"] = (ANALYSIS_AS_OF - repositories["created_at"]).dt.days.clip(lower=0)
repositories["activity_score"] = np.exp(-repositories["days_since_push"].clip(upper=3650) / 365.0)
repositories.loc[repositories["days_since_push"].isna(), "activity_score"] = 0.35
repositories["maturity_score"] = np.log1p(repositories["repository_age_days"].fillna(repositories["repository_age_days"].median()))
repositories["maturity_score"] /= max(repositories["maturity_score"].max(), 1)

def neutral_indicator(column: str) -> pd.Series:
    return repositories[column].astype("Float64").fillna(0.5).astype(float)

repositories["beginner_friendliness_score"] = (
    0.40 * neutral_indicator("has_good_first_issue") +
    0.35 * neutral_indicator("has_help_wanted") +
    0.25 * neutral_indicator("has_contributing")
)
repositories["quality_score"] = (
    0.24 * neutral_indicator("has_readme") + 0.20 * neutral_indicator("has_license") +
    0.14 * neutral_indicator("has_contributing") + 0.10 * neutral_indicator("has_code_of_conduct") +
    0.07 * neutral_indicator("has_good_first_issue") + 0.05 * neutral_indicator("has_help_wanted") +
    0.20 * repositories["maturity_score"]
).clip(0, 1)

pop_raw = 0.70 * np.log1p(repositories["stars"].fillna(0)) + 0.30 * np.log1p(repositories["forks"].fillna(0))
repositories["popularity_score"] = MinMaxScaler().fit_transform(pop_raw.to_numpy().reshape(-1, 1)).ravel()
repositories["open_issue_activity"] = MinMaxScaler().fit_transform(np.log1p(repositories["open_issues"].fillna(0)).to_numpy().reshape(-1, 1)).ravel()
print(repositories[["activity_score", "quality_score", "popularity_score", "beginner_friendliness_score"]].describe().round(3))


       activity_score  quality_score  popularity_score  beginner_friendliness_score
count         796.000        796.000           796.000                      796.000
mean            0.686          0.633             0.341                        0.498
std             0.401          0.109             0.202                        0.059
min             0.000          0.382             0.000                        0.375
25%             0.277          0.587             0.185                        0.500
50%             0.997          0.653             0.296                        0.500
75%             1.000          0.680             0.467                        0.500
max             1.000          0.937             1.000                        0.625


## 17. Candidate Generation

The sampled catalog is deliberately bounded. Eligible candidates exclude archived, disabled, known forks, zero-size repositories, metadata-empty repositories, and projects known to be inactive for over five years. Per-user ranking additionally excludes every repository used in that profile. Unknown fallback fields are not treated as negative evidence.


In [12]:
meaningful = repositories["text_full"].str.len().gt(2)
not_archived = ~repositories["archived"].fillna(False)
not_disabled = ~repositories["disabled"].fillna(False)
not_fork = ~repositories["fork"].fillna(False)
not_empty = repositories["repository_size"].isna() | repositories["repository_size"].gt(0)
recent_enough = repositories["days_since_push"].isna() | repositories["days_since_push"].le(1825)
repositories["eligible"] = meaningful & not_archived & not_disabled & not_fork & not_empty & recent_enough
eligible_ids = set(repositories.loc[repositories["eligible"], "repository_id"].astype(int))
print({"catalog": len(repositories), "eligible": len(eligible_ids), "excluded": len(repositories) - len(eligible_ids), "scope_note": "sample catalog only—not all GitHub repositories"})


{'catalog': 796, 'eligible': 685, 'excluded': 111, 'scope_note': 'sample catalog only—not all GitHub repositories'}


## 18. Random Baseline

The random baseline samples eligible unseen repositories with a deterministic developer-specific seed. It is averaged at the user level, not allowed to recommend seen items, and provides a minimal sanity check.


## 19. Popularity Baseline

The popularity baseline ranks unseen items by a normalized blend of log stars and log forks, with recent activity used only as a deterministic tie breaker. It tests whether globally famous projects alone explain retrieval.


## 20. Language Baseline

The language baseline ranks by the developer's weighted primary-language preferences. Byte distributions were unavailable in reduced mode, so this is honestly a primary-language fallback rather than a claimed full language-vector cosine.


## 21. Content-Based Recommender

TF–IDF uses English stop words, sublinear term frequency, Unicode accent stripping, technical-token preservation, and tested unigram/bigram and feature-limit variants. Repository item features may include held-out items because item metadata is public at ranking time; held-out interactions never enter the developer profile. Configuration is selected on validation NDCG@10, never test performance.


In [13]:
repo_pos = {int(repo_id): pos for pos, repo_id in enumerate(repositories["repository_id"])}

TFIDF_CONFIGS = {
    "description_unigram_8000": {"text_col": "text_description", "ngram_range": (1, 1), "max_features": 8_000},
    "description_readme_unigram_12000": {"text_col": "text_full", "ngram_range": (1, 1), "max_features": 12_000},
    "description_readme_bigram_5000": {"text_col": "text_full", "ngram_range": (1, 2), "max_features": 5_000},
    "description_readme_bigram_15000": {"text_col": "text_full", "ngram_range": (1, 2), "max_features": 15_000},
}

def fit_tfidf(config: Mapping[str, Any]) -> tuple[TfidfVectorizer, sparse.csr_matrix]:
    vectorizer = TfidfVectorizer(
        stop_words="english", min_df=2, max_df=0.95, sublinear_tf=True,
        strip_accents="unicode", token_pattern=r"(?u)\b[a-zA-Z][\w.+#-]{1,}\b",
        ngram_range=config["ngram_range"], max_features=config["max_features"], dtype=np.float32,
    )
    matrix = vectorizer.fit_transform(repositories[config["text_col"]].fillna(""))
    return vectorizer, matrix.tocsr()

def profile_inputs(developer_id: int, labels: set[str], weight_overrides: Mapping[str, float] | None = None) -> tuple[np.ndarray, np.ndarray]:
    history = interactions[(interactions["developer_id"] == developer_id) & interactions["split"].isin(labels)].copy()
    history = history[history["repository_id"].isin(repo_pos)]
    if weight_overrides:
        history["effective_weight"] = history["interaction_type"].map(weight_overrides).fillna(history["interaction_weight"])
    else:
        history["effective_weight"] = history["interaction_weight"]
    positions = np.array([repo_pos[int(repo_id)] for repo_id in history["repository_id"]], dtype=int)
    return positions, history["effective_weight"].fillna(1).to_numpy(dtype=float)

def component_scores(developer_id: int, matrix: sparse.csr_matrix, profile_labels: set[str], weight_overrides: Mapping[str, float] | None = None) -> pd.DataFrame:
    positions, weights = profile_inputs(developer_id, profile_labels, weight_overrides)
    if not len(positions):
        profile = sparse.csr_matrix((1, matrix.shape[1]))
    else:
        profile = sparse.csr_matrix(matrix[positions].multiply(weights[:, None]).sum(axis=0) / max(weights.sum(), 1e-9))
    content = linear_kernel(profile, matrix).ravel()

    history_ids = interactions[(interactions["developer_id"] == developer_id) & interactions["split"].isin(profile_labels)]["repository_id"]
    history = repositories[repositories["repository_id"].isin(history_ids)].copy()
    history_weights = interactions[(interactions["developer_id"] == developer_id) & interactions["split"].isin(profile_labels)].set_index("repository_id")["interaction_weight"]
    language_counts: Counter[str] = Counter()
    topic_counts: Counter[str] = Counter()
    for row in history.itertuples():
        weight = float(history_weights.get(row.repository_id, 1.0))
        if pd.notna(row.primary_language): language_counts[str(row.primary_language).lower()] += weight
        for topic in row.topic_list: topic_counts[topic] += weight
    max_language = max(language_counts.values(), default=1.0)
    max_topic = max(topic_counts.values(), default=1.0)
    language_scores = repositories["primary_language"].map(lambda x: language_counts.get(str(x).lower(), 0.0) / max_language if pd.notna(x) else 0.0).to_numpy(float)
    topic_scores = repositories["topic_list"].map(lambda topics: float(np.mean([topic_counts.get(t, 0.0) / max_topic for t in topics])) if topics else 0.0).to_numpy(float)
    return pd.DataFrame({
        "repository_id": repositories["repository_id"].to_numpy(),
        "content_similarity": np.clip(content, 0, 1),
        "language_score": np.clip(language_scores, 0, 1),
        "topic_score": np.clip(topic_scores, 0, 1),
        "activity_score": repositories["activity_score"].to_numpy(float),
        "quality_score": repositories["quality_score"].to_numpy(float),
        "popularity_score": repositories["popularity_score"].to_numpy(float),
    })

def rank_user(
    developer_id: int, model: str, matrix: sparse.csr_matrix, profile_labels: set[str], exclude_labels: set[str],
    top_k: int = 10, weights: Mapping[str, float] | None = None, weight_overrides: Mapping[str, float] | None = None,
    candidate_limit: int | None = None, recency_days: int = 1825, remove_top_popular: bool = False,
) -> pd.DataFrame:
    scores = component_scores(developer_id, matrix, profile_labels, weight_overrides)
    seen = set(interactions[(interactions["developer_id"] == developer_id) & interactions["split"].isin(exclude_labels)]["repository_id"].astype(int))
    allowed = repositories["eligible"].to_numpy().copy()
    allowed &= repositories["repository_id"].map(lambda x: int(x) not in seen).to_numpy()
    allowed &= (repositories["days_since_push"].isna() | repositories["days_since_push"].le(recency_days)).to_numpy()
    if remove_top_popular:
        allowed &= repositories["popularity_score"].lt(repositories["popularity_score"].quantile(.99)).to_numpy()
    scores = scores.loc[allowed].copy()
    if model == "random":
        scores["final_score"] = np.random.default_rng(RANDOM_SEED + developer_id).random(len(scores))
    elif model == "popularity":
        scores["final_score"] = 0.9 * scores["popularity_score"] + 0.1 * scores["activity_score"]
    elif model == "language":
        scores["final_score"] = scores["language_score"]
    elif model == "content":
        scores["final_score"] = scores["content_similarity"]
    elif model == "hybrid":
        assert weights is not None
        scores["final_score"] = sum(weights[name] * scores[name] for name in weights)
    else:
        raise ValueError(model)
    if candidate_limit and len(scores) > candidate_limit:
        prefilter = (0.45 * scores["content_similarity"] + 0.25 * scores["language_score"] + 0.15 * scores["topic_score"] + 0.15 * scores["activity_score"])
        scores = scores.loc[prefilter.nlargest(candidate_limit).index]
    scores = scores.sort_values(["final_score", "quality_score", "repository_id"], ascending=[False, False, True]).head(top_k)
    return scores.merge(repositories[["repository_id", "full_name", "repository_url", "primary_language", "topic_list", "has_contributing", "has_good_first_issue", "has_help_wanted"]], on="repository_id", how="left")

def ranking_metrics(recommendations: Mapping[int, Sequence[int]], relevant: Mapping[int, set[int]], matrix: sparse.csr_matrix, k: int = 10) -> dict[str, float]:
    per_user = []
    recommended_union: set[int] = set()
    diversities, novelties = [], []
    for developer_id, ranked in recommendations.items():
        ranked = list(ranked)[:k]; rel = relevant[developer_id]
        hits = np.array([repo_id in rel for repo_id in ranked], dtype=float)
        recommended_union.update(ranked)
        precision5 = hits[:5].sum() / 5
        precision10 = hits[:10].sum() / 10
        recall5 = hits[:5].sum() / max(len(rel), 1)
        recall10 = hits[:10].sum() / max(len(rel), 1)
        ap = sum(hits[i] * hits[: i + 1].mean() for i in range(min(k, len(hits)))) / max(min(len(rel), k), 1)
        dcg = sum(hits[i] / np.log2(i + 2) for i in range(min(k, len(hits))))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), k)))
        ranks = np.flatnonzero(hits)
        per_user.append({"precision_at_5": precision5, "precision_at_10": precision10, "recall_at_5": recall5, "recall_at_10": recall10, "hit_rate_at_5": float(hits[:5].any()), "hit_rate_at_10": float(hits[:10].any()), "map_at_10": ap, "ndcg_at_10": dcg / max(idcg, 1e-9), "mrr": 1 / (ranks[0] + 1) if len(ranks) else 0.0})
        positions = [repo_pos[item] for item in ranked if item in repo_pos]
        if len(positions) > 1:
            sim = cosine_similarity(matrix[positions]); upper = sim[np.triu_indices_from(sim, k=1)]
            diversities.append(float(1 - upper.mean()))
        pop = repositories.set_index("repository_id").loc[ranked, "popularity_score"] if ranked else pd.Series(dtype=float)
        novelties.append(float((1 - pop).mean()) if len(pop) else 0.0)
    aggregate = pd.DataFrame(per_user).mean().to_dict()
    aggregate.update({"catalog_coverage": len(recommended_union) / max(len(eligible_ids), 1), "intra_list_diversity": float(np.mean(diversities)) if diversities else 0.0, "novelty": float(np.mean(novelties)) if novelties else 0.0, "evaluated_users": len(per_user)})
    return {key: float(value) for key, value in aggregate.items()}

def evaluate(model: str, matrix: sparse.csr_matrix, relevant_label: str, profile_labels: set[str], exclude_labels: set[str], users: Sequence[int], weights: Mapping[str, float] | None = None, **rank_kwargs: Any) -> dict[str, float]:
    recs, relevant = {}, {}
    for developer_id in users:
        relevant[developer_id] = set(interactions[(interactions["developer_id"] == developer_id) & (interactions["split"] == relevant_label)]["repository_id"].astype(int))
        ranked = rank_user(developer_id, model, matrix, profile_labels, exclude_labels, top_k=10, weights=weights, **rank_kwargs)
        recs[developer_id] = ranked["repository_id"].astype(int).tolist()
    return ranking_metrics(recs, relevant, matrix)

tfidf_models = {name: fit_tfidf(config) for name, config in TFIDF_CONFIGS.items()}
validation_rows = []
for name, (_, matrix) in tfidf_models.items():
    result = evaluate("content", matrix, "validation", {"train"}, {"train"}, eval_users)
    validation_rows.append({"configuration": name, **result})
tfidf_validation = pd.DataFrame(validation_rows).sort_values("ndcg_at_10", ascending=False)
best_tfidf_name = str(tfidf_validation.iloc[0]["configuration"])
final_vectorizer, final_matrix = tfidf_models[best_tfidf_name]
print(tfidf_validation[["configuration", "hit_rate_at_10", "ndcg_at_10", "catalog_coverage"]].round(4).to_string(index=False))
print("Selected without test access:", best_tfidf_name, "features=", final_matrix.shape[1])


                   configuration  hit_rate_at_10  ndcg_at_10  catalog_coverage
description_readme_unigram_12000             0.6      0.1214            0.0599
  description_readme_bigram_5000             0.4      0.1017            0.0599
        description_unigram_8000             0.2      0.0632            0.0599
 description_readme_bigram_15000             0.4      0.0535            0.0584
Selected without test access: description_readme_unigram_12000 features= 4572


## 22. Collaborative-Filtering Feasibility

Collaborative filtering is implemented only when interaction structure supports it. The gate requires at least 30 developers, meaningful item co-support, and adequate matrix density. A numerically nonzero density is not enough when nearly every repository is observed by just one of five users.


In [14]:
user_count = interactions["developer_id"].nunique()
repo_count = interactions["repository_id"].nunique()
interaction_count = len(interactions)
item_support = interactions.groupby("repository_id")["developer_id"].nunique()
cf_stats = {
    "developers": user_count,
    "repositories_with_interactions": repo_count,
    "interactions": interaction_count,
    "avg_interactions_per_developer": interaction_count / max(user_count, 1),
    "avg_interactions_per_repository": interaction_count / max(repo_count, 1),
    "matrix_density": interaction_count / max(user_count * repo_count, 1),
    "cold_start_users_pct": 100 * (activity_counts < 5).mean(),
    "cold_start_repositories_pct": 100 * (item_support < 2).mean(),
    "items_with_2plus_users_pct": 100 * (item_support >= 2).mean(),
}
CF_VIABLE = user_count >= 30 and (item_support >= 2).mean() >= 0.10 and cf_stats["matrix_density"] >= 0.005
print(pd.Series(cf_stats).round(4).to_string())
print("Collaborative filtering viable:", CF_VIABLE, "— content-plus-quality hybrid used because user/item overlap is insufficient.")


developers                           5.0000
repositories_with_interactions     446.0000
interactions                       449.0000
avg_interactions_per_developer      89.8000
avg_interactions_per_repository      1.0067
matrix_density                       0.2013
cold_start_users_pct                 0.0000
cold_start_repositories_pct         99.3274
items_with_2plus_users_pct           0.6726
Collaborative filtering viable: False — content-plus-quality hybrid used because user/item overlap is insufficient.


## 23. Hybrid Ranking Model

Because collaborative filtering fails the quantitative gate, the final family combines content, language, topic, activity, quality, and capped popularity. The exact formula is `sum(weight × component)` and every candidate uses the same normalized component scale. Weight sets are selected on validation NDCG@10 only.


In [15]:
WEIGHT_CONFIGS = {
    "initial_balanced": {"content_similarity": .45, "language_score": .20, "topic_score": .15, "activity_score": .08, "quality_score": .08, "popularity_score": .04},
    "content_heavy": {"content_similarity": .55, "language_score": .15, "topic_score": .10, "activity_score": .08, "quality_score": .08, "popularity_score": .04},
    "contribution_ready": {"content_similarity": .35, "language_score": .15, "topic_score": .15, "activity_score": .12, "quality_score": .18, "popularity_score": .05},
    "discovery": {"content_similarity": .40, "language_score": .18, "topic_score": .12, "activity_score": .12, "quality_score": .08, "popularity_score": .10},
    "low_popularity": {"content_similarity": .48, "language_score": .20, "topic_score": .14, "activity_score": .09, "quality_score": .08, "popularity_score": .01},
}
weight_rows = []
for name, weights in WEIGHT_CONFIGS.items():
    assert abs(sum(weights.values()) - 1) < 1e-9
    result = evaluate("hybrid", final_matrix, "validation", {"train"}, {"train"}, eval_users, weights=weights)
    weight_rows.append({"weight_configuration": name, **result})
weight_validation = pd.DataFrame(weight_rows).sort_values(["ndcg_at_10", "catalog_coverage"], ascending=False)
best_weight_name = str(weight_validation.iloc[0]["weight_configuration"])
FINAL_WEIGHTS = WEIGHT_CONFIGS[best_weight_name]
print(weight_validation[["weight_configuration", "hit_rate_at_10", "ndcg_at_10", "catalog_coverage", "novelty"]].round(4).to_string(index=False))
print("Selected weights:", best_weight_name, FINAL_WEIGHTS)


weight_configuration  hit_rate_at_10  ndcg_at_10  catalog_coverage  novelty
    initial_balanced             0.6      0.1072            0.0715   0.4963
  contribution_ready             0.6      0.1009            0.0672   0.4020
           discovery             0.6      0.0983            0.0672   0.3841
       content_heavy             0.6      0.0820            0.0730   0.5258
      low_popularity             0.2      0.0339            0.0730   0.5746
Selected weights: initial_balanced {'content_similarity': 0.45, 'language_score': 0.2, 'topic_score': 0.15, 'activity_score': 0.08, 'quality_score': 0.08, 'popularity_score': 0.04}


## 24. Offline Evaluation Design

For every committed REST-star history, training precedes validation, which precedes the five most recent test stars. Held-out repositories never enter the profile and remain eligible candidates. A seeded leave-five-out fallback remains available for future records without timestamps and is explicitly labelled nonchronological. Final test scoring is run once after text and weight selection; training plus validation history forms the final profile.


## 25. Recommendation Metrics

- **Precision@K:** fraction of the top-K recommendations that are relevant.
- **Recall@K:** fraction of held-out relevant repositories retrieved in the top K.
- **Hit Rate@K:** whether at least one relevant item appears in the top K.
- **MAP@K:** mean precision accumulated at relevant ranks.
- **NDCG@K:** rank-sensitive gain that rewards relevant repositories near the top.
- **MRR:** reciprocal rank of the first relevant recommendation.
- **Coverage:** unique recommended items divided by the eligible sampled catalog.
- **Diversity:** one minus mean pairwise TF–IDF cosine similarity within each list.
- **Novelty:** mean one-minus-normalized-popularity, rewarding long-tail exposure.

Metrics are macro-averaged across developers so prolific histories do not dominate.


In [16]:
model_rows = []
for model in ["random", "popularity", "language", "content", "hybrid"]:
    result = evaluate(model, final_matrix, "test", {"train", "validation"}, {"train", "validation"}, eval_users, weights=FINAL_WEIGHTS if model == "hybrid" else None)
    model_rows.append({"model": model, **result})
evaluation_metrics = pd.DataFrame(model_rows)
best_model = str(evaluation_metrics.sort_values(["ndcg_at_10", "hit_rate_at_10", "catalog_coverage"], ascending=False).iloc[0]["model"])
evaluation_metrics["best_by_ndcg_at_10"] = evaluation_metrics["model"].eq(best_model)
evaluation_metrics.to_csv(OUTPUT_DIR / "evaluation_metrics.csv", index=False)
model_comparison = evaluation_metrics.copy()
model_comparison["tfidf_configuration"] = best_tfidf_name
model_comparison["hybrid_weight_configuration"] = best_weight_name
model_comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
print(evaluation_metrics.round(4).to_string(index=False))


     model  precision_at_5  precision_at_10  recall_at_5  recall_at_10  hit_rate_at_5  hit_rate_at_10  map_at_10  ndcg_at_10    mrr  catalog_coverage  intra_list_diversity  novelty  evaluated_users  best_by_ndcg_at_10
    random            0.00             0.00         0.00          0.00            0.0             0.0     0.0000      0.0000 0.0000            0.0730                0.9864   0.6423              5.0               False
popularity            0.04             0.02         0.04          0.04            0.2             0.2     0.0400      0.0678 0.2000            0.0263                0.8946   0.1161              5.0               False
  language            0.08             0.04         0.08          0.08            0.4             0.4     0.0800      0.1357 0.4000            0.0701                0.9300   0.4181              5.0                True
   content            0.04             0.04         0.04          0.08            0.2             0.2     0.0180      0.0476 0.0

## 26. Model Comparison

NDCG@10 is the primary selection metric because this is a ranked retrieval task. Recall/hit rate show retrieval, while coverage/diversity/novelty expose concentration trade-offs. With only five evaluated users, differences are descriptive sample results, not confidence-qualified population claims.


In [17]:
fig, ax = plt.subplots(figsize=(8, 4.8))
comparison_plot = evaluation_metrics.set_index("model")[["ndcg_at_10", "hit_rate_at_10", "catalog_coverage"]]
comparison_plot.plot(kind="bar", ax=ax, color=["#264653", "#2a9d8f", "#e9c46a"])
ax.set_title("Offline model comparison"); ax.set_xlabel("model"); ax.set_ylabel("metric value"); ax.set_ylim(0, max(1.0, comparison_plot.max().max() * 1.1)); ax.tick_params(axis="x", rotation=25)
fig.tight_layout(); fig.savefig(FIGURE_DIR / "model_comparison.png", dpi=150); plt.close(fig)
print("Best measured model by test NDCG@10:", best_model)


Best measured model by test NDCG@10: language


## 27. Recommendation Examples

Production-style recommendations rebuild each profile from every observed interaction and exclude all observed repositories. The callable `recommend_for_user(username, top_k)` works for sampled users; a truly new username follows the cold-start path in section 31 until public history is collected.


In [18]:
username_to_id = dict(zip(developers["username"].str.lower(), developers["developer_id"].astype(int)))

def recommendation_explanation(developer_id: int, row: pd.Series) -> str:
    history_ids = interactions[interactions["developer_id"] == developer_id]["repository_id"]
    history = repositories[repositories["repository_id"].isin(history_ids)]
    languages = history["primary_language"].dropna().str.lower().value_counts()
    topics = Counter(topic for values in history["topic_list"] for topic in values)
    reasons: list[str] = []
    language = str(row.get("primary_language", "")).lower()
    if language and language in languages.index and row["language_score"] > 0:
        reasons.append(f"{row['primary_language']} is a strong language in the public profile")
    overlap = sorted(set(row.get("topic_list", [])) & set(topics), key=lambda t: topics[t], reverse=True)[:2]
    if overlap and row["topic_score"] > 0:
        reasons.append("topics overlap with " + " and ".join(overlap))
    if row["content_similarity"] >= 0.20:
        reasons.append("README/description terms resemble previously observed projects")
    if row["activity_score"] >= 0.70:
        reasons.append("the project has been active recently")
    if bool(row.get("has_contributing")) if pd.notna(row.get("has_contributing")) else False:
        reasons.append("it includes contribution guidance")
    if bool(row.get("has_good_first_issue")) if pd.notna(row.get("has_good_first_issue")) else False:
        reasons.append("good-first-issue labels were observed")
    if row["popularity_score"] < 0.30 and row["content_similarity"] >= 0.15:
        reasons.append("it is a less-popular but profile-relevant discovery")
    if not reasons:
        reasons.append("its calculated language, activity, and quality signals fit the profile")
    return "; ".join(reasons[:3]).capitalize() + "."

def recommend_for_user(username: str, top_k: int = 10) -> pd.DataFrame:
    key = username.lower()
    if key not in username_to_id:
        raise ValueError("Unknown sampled user; collect public history or use the cold-start questionnaire in section 31.")
    developer_id = username_to_id[key]
    ranked = rank_user(developer_id, "hybrid", final_matrix, {"train", "validation", "test"}, {"train", "validation", "test"}, top_k=top_k, weights=FINAL_WEIGHTS)
    ranked["username"] = username
    ranked["rank"] = np.arange(1, len(ranked) + 1)
    ranked["explanation"] = ranked.apply(lambda row: recommendation_explanation(developer_id, row), axis=1)
    return ranked[["username", "full_name", "repository_url", "rank", "final_score", "content_similarity", "language_score", "topic_score", "activity_score", "quality_score", "popularity_score", "explanation"]].rename(columns={"full_name": "repository_full_name"})

recommendations = pd.concat([recommend_for_user(username, 10) for username in developers["username"]], ignore_index=True)
recommendations.to_csv(OUTPUT_DIR / "recommendations.csv", index=False)
print(recommendations.groupby("username").head(3).to_string(index=False))


    username              repository_full_name                                       repository_url  rank  final_score  content_similarity  language_score  topic_score  activity_score  quality_score  popularity_score                                                                                                                                       explanation
sindresorhus           geodienst/lighthousemap           https://github.com/geodienst/lighthousemap     1     0.420821            0.158867             1.0     0.000000             1.0       0.684237          0.364807                                                      Javascript is a strong language in the public profile; the project has been active recently.
sindresorhus      Baskerville42/outage-data-ua      https://github.com/Baskerville42/outage-data-ua     2     0.410272            0.158867             1.0     0.000000             1.0       0.627429          0.214691 Javascript is a strong language in the public profile; th

## 28. Explainability

Explanations are deterministic feature attributions, not generated filler. A reason is emitted only when its underlying language/topic/content/activity/quality/popularity condition is true. Scores remain available beside the prose for auditability.


In [19]:
example_user = developers.sort_values("username").iloc[0]["username"]
print(recommend_for_user(str(example_user), 5)[["rank", "repository_full_name", "final_score", "content_similarity", "language_score", "topic_score", "explanation"]].to_string(index=False))


 rank              repository_full_name  final_score  content_similarity  language_score  topic_score                                                                                                                           explanation
    1 bcherny/json-schema-to-typescript     0.431135            0.038198             1.0         0.30          Typescript is a strong language in the public profile; topics overlap with typescript; the project has been active recently.
    2         get-convex/convex-backend     0.411325            0.045094             1.0         0.14 Typescript is a strong language in the public profile; topics overlap with typescript and rust; the project has been active recently.
    3                       KaTeX/KaTeX     0.405644            0.037262             1.0         0.04 Typescript is a strong language in the public profile; topics overlap with math and javascript; the project has been active recently.
    4                    N3rdmade/TBCPL     0.403706    

## 29. Error Analysis

Errors are inspected by user, source method, missed held-out repositories, popularity domination, list repetition, inactivity, and missing content. A held-out star is a noisy relevance label: failure may indicate model weakness, changing interests, or a star made for reference rather than affinity.


In [20]:
error_rows = []
for developer_id in eval_users:
    ranked = rank_user(developer_id, "hybrid", final_matrix, {"train", "validation"}, {"train", "validation"}, top_k=len(eligible_ids), weights=FINAL_WEIGHTS)
    rank_map = {int(repo_id): rank + 1 for rank, repo_id in enumerate(ranked["repository_id"])}
    heldout = interactions[(interactions["developer_id"] == developer_id) & (interactions["split"] == "test")]["repository_id"].astype(int)
    ranks = [rank_map.get(repo_id, np.inf) for repo_id in heldout]
    error_rows.append({"developer_id": developer_id, "username": developers.set_index("developer_id").loc[developer_id, "username"], "split_method": split_method[developer_id], "best_heldout_rank": min(ranks), "mean_finite_heldout_rank": float(np.mean([r for r in ranks if np.isfinite(r)])) if any(np.isfinite(ranks)) else np.inf, "hit_at_10": any(r <= 10 for r in ranks)})
error_analysis = pd.DataFrame(error_rows).sort_values(["hit_at_10", "best_heldout_rank"])
weakest = error_analysis.iloc[0]
pop_dominated = recommendations.sort_values(["popularity_score", "content_similarity"], ascending=[False, True]).iloc[0]
long_tail = recommendations.sort_values(["popularity_score", "content_similarity"], ascending=[True, False]).iloc[0]
print(error_analysis.to_string(index=False))
print("Weakest-profile example:", weakest.to_dict())
print("Popularity-dominated example:", pop_dominated[["username", "repository_full_name", "popularity_score", "content_similarity"]].to_dict())
print("Long-tail example:", long_tail[["username", "repository_full_name", "popularity_score", "content_similarity"]].to_dict())
print("Diagnosis: missing README/topics weaken content; broad or changing interests reduce held-out predictability; fallback users lack chronology; popularity remains bounded by its selected weight.")


 developer_id     username  split_method  best_heldout_rank  mean_finite_heldout_rank  hit_at_10
       810438      gaearon chronological                 11                     98.80      False
       781659      jakevdp chronological                 14                    114.40      False
       170270 sindresorhus chronological                 61                     65.60      False
       241138     karpathy chronological                 67                    156.20      False
         4196       hadley chronological                  3                    240.75       True


Weakest-profile example: {'developer_id': 810438, 'username': 'gaearon', 'split_method': 'chronological', 'best_heldout_rank': 11, 'mean_finite_heldout_rank': 98.8, 'hit_at_10': False}
Popularity-dominated example: {'username': 'gaearon', 'repository_full_name': 'anomalyco/opencode', 'popularity_score': 0.918979728266303, 'content_similarity': 0.03244986315783302}
Long-tail example: {'username': 'sindresorhus', 'repository_full_name': 'AnEntrypoint/gm', 'popularity_score': 0.11891054484706523, 'content_similarity': 0.1588670533709228}
Diagnosis: missing README/topics weaken content; broad or changing interests reduce held-out predictability; fallback users lack chronology; popularity remains bounded by its selected weight.


## 30. Robustness Testing

Robustness checks are calculated rather than narrated: description-only versus README-enhanced text, unigrams/bigrams, two feature limits, alternative ranking/popularity weights, equal interaction weights, minimum-history thresholds, candidate-pool limits, recency filtering, highly popular repository removal, and both K=5 and K=10 metrics. Selection remains validation-only; test variants diagnose sensitivity and do not revise the final model.


In [21]:
robust_rows = []
for _, row in tfidf_validation.iterrows():
    robust_rows.append({"scenario": "tfidf:" + row["configuration"], "split": "validation", "ndcg_at_10": row["ndcg_at_10"], "hit_rate_at_10": row["hit_rate_at_10"], "catalog_coverage": row["catalog_coverage"]})
for _, row in weight_validation.iterrows():
    robust_rows.append({"scenario": "weights:" + row["weight_configuration"], "split": "validation", "ndcg_at_10": row["ndcg_at_10"], "hit_rate_at_10": row["hit_rate_at_10"], "catalog_coverage": row["catalog_coverage"]})

test_scenarios = {
    "final_default": {},
    "equal_interaction_weights": {"weight_overrides": {"owned": 1, "starred": 1, "forked": 1, "contributed": 1}},
    "candidate_pool_100": {"candidate_limit": 100},
    "candidate_pool_300": {"candidate_limit": 300},
    "recency_730_days": {"recency_days": 730},
    "remove_top_1pct_popular": {"remove_top_popular": True},
}
for scenario, kwargs in test_scenarios.items():
    result = evaluate("hybrid", final_matrix, "test", {"train", "validation"}, {"train", "validation"}, eval_users, weights=FINAL_WEIGHTS, **kwargs)
    robust_rows.append({"scenario": scenario, "split": "test_diagnostic", "ndcg_at_10": result["ndcg_at_10"], "hit_rate_at_10": result["hit_rate_at_10"], "catalog_coverage": result["catalog_coverage"]})
for threshold in [20, 50, 80]:
    users = [dev for dev in eval_users if (interactions[(interactions["developer_id"] == dev) & (interactions["split"] == "train")].shape[0] >= threshold)]
    if users:
        result = evaluate("hybrid", final_matrix, "test", {"train", "validation"}, {"train", "validation"}, users, weights=FINAL_WEIGHTS)
        robust_rows.append({"scenario": f"minimum_history_{threshold}", "split": "test_diagnostic", "ndcg_at_10": result["ndcg_at_10"], "hit_rate_at_10": result["hit_rate_at_10"], "catalog_coverage": result["catalog_coverage"]})
robustness_results = pd.DataFrame(robust_rows)
print(robustness_results.round(4).to_string(index=False))
print("K sensitivity is visible directly in precision/recall/hit-rate columns at K=5 and K=10 in evaluation_metrics.csv.")


                              scenario           split  ndcg_at_10  hit_rate_at_10  catalog_coverage
tfidf:description_readme_unigram_12000      validation      0.1214            0.60            0.0599
  tfidf:description_readme_bigram_5000      validation      0.1017            0.40            0.0599
        tfidf:description_unigram_8000      validation      0.0632            0.20            0.0599
 tfidf:description_readme_bigram_15000      validation      0.0535            0.40            0.0584
              weights:initial_balanced      validation      0.1072            0.60            0.0715
            weights:contribution_ready      validation      0.1009            0.60            0.0672
                     weights:discovery      validation      0.0983            0.60            0.0672
                 weights:content_heavy      validation      0.0820            0.60            0.0730
                weights:low_popularity      validation      0.0339            0.20         

## 31. Cold-Start Strategy

**New developer:** ask for preferred languages and topics, project type, skill level, learning-versus-contributing goal, and activity tolerance; turn answers into the same language/topic/activity feature vector and recommend active eligible repositories. **New repository:** score description, README, topics, languages, recency, license, contribution guidance, code of conduct, and issue labels without waiting for interactions. This avoids forcing collaborative filtering where no history exists.


## 32. Feedback-Loop Design

Future explicit feedback values are `interested`, `not_interested`, `save_for_later`, `already_familiar`, `too_advanced`, and `too_inactive`. Store only what is necessary using:

`username, repository_id, feedback_type, feedback_timestamp, recommendation_score, model_version`

Positive and negative feedback should be separately weighted, monitored for position/exposure bias, versioned, and never interpreted as a sensitive personal attribute.


## 33. Limitations

The five-user convenience sample is too small for stable population claims or collaborative filtering. The catalog is search- and history-derived rather than all of GitHub. README checks cover a subset; language byte distributions and issue-label signals are missing in reduced mode. Stars are imperfect relevance labels. Public visibility does not remove the obligation to minimize profiling and provide user control.


In [22]:
print({"sample_users": len(developers), "eligible_catalog": len(eligible_ids), "chronological_users": sum(v == "chronological" for v in split_method.values()), "fallback_users": sum(v != "chronological" for v in split_method.values()), "collaborative_filtering_used": CF_VIABLE, "api_failures": len(manifest.get("api_failures", []))})


{'sample_users': 5, 'eligible_catalog': 685, 'chronological_users': 5, 'fallback_users': 0, 'collaborative_filtering_used': False, 'api_failures': 0}


## 34. Final Conclusions

The code below answers every decision question directly from `evaluation_metrics.csv`; it does not assume the hybrid wins. “Outperformed” means higher measured NDCG@10 on this sample, not statistical significance.


In [23]:
metric_lookup = evaluation_metrics.set_index("model")
content_ndcg = metric_lookup.loc["content", "ndcg_at_10"]
answers = {
    "content_outperformed_random": bool(content_ndcg > metric_lookup.loc["random", "ndcg_at_10"]),
    "content_outperformed_popularity": bool(content_ndcg > metric_lookup.loc["popularity", "ndcg_at_10"]),
    "content_outperformed_language": bool(content_ndcg > metric_lookup.loc["language", "ndcg_at_10"]),
    "best_model": best_model,
    "best_ndcg_at_10": float(metric_lookup.loc[best_model, "ndcg_at_10"]),
    "best_catalog_coverage": float(metric_lookup.loc[best_model, "catalog_coverage"]),
    "readme_improved_validation": bool(tfidf_validation[tfidf_validation["configuration"].str.contains("readme")]["ndcg_at_10"].max() > tfidf_validation[tfidf_validation["configuration"].eq("description_unigram_8000")]["ndcg_at_10"].iloc[0]),
    "popularity_bias_present": bool(metric_lookup.loc["popularity", "novelty"] < metric_lookup.loc["hybrid", "novelty"]),
    "largest_limitation": "five-user convenience sample with a bounded search/history-derived catalog",
}
print(json.dumps(answers, indent=2))
print("Most useful feature evidence is validation sensitivity across TF-IDF and hybrid weight configurations; limited-history performance is shown in robustness_results, not generalized beyond this sample.")


{
  "content_outperformed_random": true,
  "content_outperformed_popularity": false,
  "content_outperformed_language": false,
  "best_model": "language",
  "best_ndcg_at_10": 0.13566408210944644,
  "best_catalog_coverage": 0.07007299270072993,
  "readme_improved_validation": true,
  "popularity_bias_present": true,
  "largest_limitation": "five-user convenience sample with a bounded search/history-derived catalog"
}
Most useful feature evidence is validation sensitivity across TF-IDF and hybrid weight configurations; limited-history performance is shown in robustness_results, not generalized beyond this sample.


## 35. Future Improvements

Collect 50–150 consent-aware public developers and 1,000–5,000 repositories with an authenticated token; fetch complete language bytes and issue labels; add contribution events only where reliably attributable; use more time windows and bootstrap confidence intervals; learn ranking weights from explicit feedback; measure exposure fairness and popularity bias; and reconsider item-item or implicit-feedback factorization only after user/item overlap passes the documented gate.


In [24]:
joblib.dump(final_vectorizer, MODEL_DIR / "tfidf_vectorizer.joblib")
joblib.dump(final_matrix, MODEL_DIR / "repository_feature_matrix.joblib")
expected = [OUTPUT_DIR / "recommendations.csv", OUTPUT_DIR / "model_comparison.csv", OUTPUT_DIR / "evaluation_metrics.csv", OUTPUT_DIR / "data_quality_summary.csv", MODEL_DIR / "tfidf_vectorizer.joblib", MODEL_DIR / "repository_feature_matrix.joblib"]
assert all(path.exists() and path.stat().st_size > 0 for path in expected)
print("Reproducibility artifacts verified:", [str(path.relative_to(PROJECT_ROOT)) for path in expected])


Reproducibility artifacts verified: ['outputs\\recommendations.csv', 'outputs\\model_comparison.csv', 'outputs\\evaluation_metrics.csv', 'outputs\\data_quality_summary.csv', 'models\\tfidf_vectorizer.joblib', 'models\\repository_feature_matrix.joblib']
